# Image Generation in Products

**Phase 10** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-10/03-image-generation.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q openai

import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running locally — set OPENAI_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

## The Problem

A product manager wants to add AI image generation to a marketing platform. Users will describe a campaign concept in text and receive a generated image to use in social posts. The engineering team has one afternoon to spike it.

They hit five decisions immediately:

1. Which API? DALL-E 3 (OpenAI), Stable Diffusion via Replicate, Midjourney via API. These are different products with different trade-offs.
2. Content policy violations. Marketing copy can describe brand partnerships, alcohol, competitive products. Some of these will trigger refusals. What happens to the user experience when a refusal occurs?
3. Generation takes 5-20 seconds. A synchronous API call that hangs for 15 seconds in ...

## The Concept

### Provider landscape

```
Provider comparison: image generation APIs

+------------------+----------+-----------+--------------------+------------------+
| Provider         | Cost/img | P95       | Content policy     | Best for         |
|                  |          | latency   |                    |                  |
+------------------+----------+-----------+--------------------+------------------+
| DALL-E 3         | $0.040   | 12-20s    | Strict, auto-safe  | Product UX,      |
| (OpenAI)         | (1024px) |           | filter built in    | brand safety     |
+------------------+----...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 10-03: Image Generation in Products
DALL-E 3 image generation with content policy handling, retry logic,
and async service pattern demonstration.

Usage:
    python main.py                      # demo mode (no API calls)
    python main.py "a product photo"    # generate with DALL-E 3
    python main.py --demo               # force demo mode

Requirements:
    pip install openai
    Optional: pip install replicate
"""

import json
import os
import sys
import time
import threading
import urllib.request
import uuid
from pathlib import Path
from typing import Optional


# --------------------------------------------------------------------------- #
# Content policy prompt sanitizer                                              #
# --------------------------------------------------------------------------- #

SENSITIVE_KEYWORDS = [
    "violent", "nude", "explicit", "blood", "weapon",
    "realistic person", "real person", "celebrity",
]


def sanitize_prompt(prompt: str) -> str:
    """Remove known sensitive keywords and add a safety suffix."""
    cleaned = prompt
    for kw in SENSITIVE_KEYWORDS:
        cleaned = cleaned.replace(kw, "").replace(kw.title(), "")
    while "  " in cleaned:
        cleaned = cleaned.replace("  ", " ")
    return cleaned.strip() + ", professional context, brand-safe"


# --------------------------------------------------------------------------- #
# DALL-E 3 generation                                                          #
# --------------------------------------------------------------------------- #

def generate_dalle3(
    prompt: str,
    size: str = "1024x1024",
    quality: str = "standard",
    output_path: Optional[Path] = None,
    demo_mode: bool = False,
) -> dict:
    """Generate an image with DALL-E 3."""
    if demo_mode:
        return {
            "status": "success",
            "image_url": "https://example.com/placeholder.png",
            "revised_prompt": f"[DEMO] {prompt}",
            "cost_estimate": 0.040,
            "latency_seconds": 0.0,
            "provider": "dalle3-demo",
        }

    try:
        from openai import OpenAI
    except ImportError:
        raise SystemExit("Install openai: pip install openai")

    client = OpenAI()
    start = time.time()

    try:
        response = client.images.generate(
            model="dall-e-3",
            prompt=prompt,
            size=size,
            quality=quality,
            n=1,
        )
        latency = time.time() - start
        image_url = response.data[0].url
        revised_prompt = getattr(response.data[0], "revised_prompt", prompt)

        cost_map = {"1024x1024": 0.040, "1792x1024": 0.080, "1024x1792": 0.080}
        cost = cost_map.get(size, 0.040) * (2 if quality == "hd" else 1)

        if output_path:
            urllib.request.urlretrieve(image_url, output_path)
            print(f"  Saved to: {output_path}")

        return {
            "status": "success",
            "image_url": image_url,
            "revised_prompt": revised_prompt,
            "cost_estimate": cost,
            "latency_seconds": round(latency, 2),
            "provider": "dalle3",
        }

    except Exception as e:
        err = str(e)
        if "content_policy_violation" in err or "safety system" in err.lower():
            return {
                "status": "content_policy_violation",
                "error": err,
                "original_prompt": prompt,
            }
        raise


def generate_with_retry(
    prompt: str,
    output_path: Optional[Path] = None,
    demo_mode: bool = False,
) -> dict:
    """
    Generate with automatic sanitized-prompt retry on content policy violations.
    Two attempts maximum: original prompt, then sanitized prompt.
    """
    result = generate_dalle3(prompt, output_path=output_path, demo_mode=demo_mode)

    if result["status"] == "content_policy_violation":
        print("  Content policy violation on original prompt.")
        sanitized = sanitize_prompt(prompt)
        print(f"  Retrying with sanitized prompt: {sanitized[:80]}...")
        result = generate_dalle3(sanitized, output_path=output_path, demo_mode=demo_mode)
        result["was_sanitized"] = True
        result["original_prompt"] = prompt
        result["sanitized_prompt"] = sanitized

        if result["status"] == "content_policy_violation":
            print("  Second violation. Generation failed.")
            return {
                "status": "failed_content_policy",
                "error": "Could not generate safe image after sanitization",
                "original_prompt": prompt,
                "image_url": None,
            }

    return result


# --------------------------------------------------------------------------- #
# Replicate (Stable Diffusion) pattern                                         #
# --------------------------------------------------------------------------- #

def generate_replicate_sd(
    prompt: str,
    negative_prompt: str = "blurry, low quality, distorted, watermark",
    demo_mode: bool = False,
) -> dict:
    """
    Generate via Replicate SDXL. Shows open-weight alternative pattern.
    Much cheaper than DALL-E 3 (~$0.0023/image) but requires Replicate account.
    """
    if demo_mode:
        return {
            "status": "success",
            "image_url": "https://example.com/sd-placeholder.png",
            "cost_estimate": 0.0023,
            "provider": "replicate-sdxl-demo",
            "negative_prompt": negative_prompt,
        }

    try:
        import replicate
    except ImportError:
        raise SystemExit("Install replicate: pip install replicate")

    output = replicate.run(
        "stability-ai/sdxl:39ed52f2a78e934b3ba6e2a89f5b1c712de7dfea535525255b1aa35c5565e08b",
        input={
            "prompt": prompt,
            "negative_prompt": negative_prompt,
            "num_outputs": 1,
            "width": 1024,
            "height": 1024,
        },
    )
    return {
        "status": "success",
        "image_url": output[0],
        "cost_estimate": 0.0023,
        "provider": "replicate-sdxl",
    }


# --------------------------------------------------------------------------- #
# Async service simulation                                                     #
# --------------------------------------------------------------------------- #

_store: dict[str, dict] = {}


def submit_generation_async(prompt: str, demo_mode: bool = False) -> str:
    """
    Async pattern: accept prompt, return generation_id immediately.
    Background thread runs generation (use Celery/ARQ in production).
    """
    gen_id = str(uuid.uuid4())[:8]
    _store[gen_id] = {"status": "pending", "prompt": prompt}

    def _worker():
        _store[gen_id]["status"] = "generating"
        result = generate_with_retry(prompt, demo_mode=demo_mode)
        _store[gen_id].update(result)
        # In production: upload image to S3, store permanent URL, send webhook

    threading.Thread(target=_worker, daemon=True).start()
    return gen_id


def get_generation_status(gen_id: str) -> dict:
    """Poll status. In production: GET /generations/{id}"""
    return _store.get(gen_id, {"status": "not_found"})


# --------------------------------------------------------------------------- #
# Main                                                                         #
# --------------------------------------------------------------------------- #

def main():
    print("=== Lesson 10-03: Image Generation in Products ===\n")

    demo_mode = "--demo" in sys.argv or "OPENAI_API_KEY" not in os.environ
    if demo_mode:
        print("Demo mode active (set OPENAI_API_KEY to use real API)\n")

    prompt_parts = [a for a in sys.argv[1:] if not a.startswith("--")]
    prompt = " ".join(prompt_parts) or (
        "A confident product manager presenting quarterly results at a whiteboard, "
        "modern open office, soft natural lighting, professional photography style, "
        "sharp focus, high quality"
    )

    print(f"Prompt: {prompt}\n")

    # --- Direct generation ---
    print("=== 1. Direct generation with content policy retry ===")
    output_file = Path("generated_image.png")
    result = generate_with_retry(
        prompt,
        output_path=output_file if not demo_mode else None,
        demo_mode=demo_mode,
    )
    print(json.dumps(result, indent=2))

    # --- Async service pattern ---
    print("\n=== 2. Async generation service pattern ===")
    gen_id = submit_generation_async(prompt, demo_mode=demo_mode)
    print(f"Submitted. generation_id: {gen_id}")
    print("Client receives 202 Accepted; polls for status:")

    for attempt in range(8):
        status = get_generation_status(gen_id)
        current_status = status.get("status", "unknown")
        print(f"  Poll {attempt + 1}: {current_status}")
        if current_status not in ("pending", "generating"):
            break
        time.sleep(0.2 if demo_mode else 2.0)

    final = get_generation_status(gen_id)
    print(f"\nFinal: image_url = {final.get('image_url', 'N/A')}")

    # --- Provider comparison ---
    print("\n=== 3. Alternative: Replicate SD (open-weight) ===")
    sd = generate_replicate_sd(prompt, demo_mode=True)
    print(json.dumps(sd, indent=2))

    print("\n=== Provider cost comparison ===")
    providers = [
        ("DALL-E 3 (1024px standard)", 0.040),
        ("DALL-E 3 (1024px HD)", 0.080),
        ("DALL-E 2 (1024px)", 0.020),
        ("Replicate SDXL", 0.0023),
        ("Replicate Flux", 0.003),
    ]
    for name, cost in providers:
        monthly_1k = cost * 1000
        print(f"  {name:<30} ${cost:.4f}/img  ${monthly_1k:.2f}/1k imgs")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. At 500 images/day, what is the monthly cost difference between DALL-E 3 and Replicate SDXL, and what factor should drive the final decision?**

- A. $576/month difference; DALL-E 3 should always win because it is from OpenAI
- B. About $576/month difference; the decision should be driven by actual output quality for your specific use case, not vendor reputation
- C. The cost is identical because Replicate charges by compute time, not per image
- D. $576/month difference; Replicate SDXL should always win because it is open-weight

**2. Which two design flaws caused this experience?**

- A. No content policy retry logic, and using a synchronous HTTP handler for a 5-20 second operation
- B. Using DALL-E 3 instead of Stable Diffusion, and not enough max_tokens in the generation call
- C. Missing authentication middleware and too-short request timeout
- D. Content policy violation was not caught as an exception, and the image was not resized before serving

**3. Why are the images broken, and what is the correct storage pattern?**

- A. The database connection timed out; use a connection pool
- B. DALL-E response URLs are temporary and expire in roughly 1 hour; images must be downloaded and stored in your own object storage (S3/GCS) immediately after generation
- C. The OpenAI API changed the URL format; update the URL parsing code
- D. Images expire because DALL-E 3 has a rate limit on serving stored images

**4. Which approach is most likely to reduce content policy violations for alcohol marketing prompts?**

- A. Switch to a provider with more permissive content filters (Replicate with uncensored model)
- B. Pre-process prompts to reframe the content contextually (e.g., 'premium craft beer in a social setting' instead of 'beer commercial') and avoid trigger keywords
- C. Increase the max resolution to 1792x1024 so the model has more room to express the concept
- D. Add a negative prompt specifying what not to include

**5. Why is Ideogram v2 better suited for this use case than DALL-E 3?**

- A. Ideogram is cheaper per image than DALL-E 3
- B. Ideogram v2 is specifically optimized for accurate text rendering within images, making it better for graphics that embed product names and slogans
- C. DALL-E 3 cannot generate images larger than 512x512
- D. Ideogram has no content policy restrictions

**6. What information should the approval queue surface to reviewers to make their job efficient?**

- A. Only the generated image, without the original prompt, to avoid biasing the reviewer
- B. The original user prompt, the revised_prompt (what DALL-E 3 actually used), and the generated image side by side
- C. The generated image and the generation cost, so reviewers can reject expensive images
- D. Only the revised_prompt and the image, because the original prompt is irrelevant after rewriting

**7. What additional metric would best capture whether the image generation feature is actually valuable to the business?**

- A. Average image file size in kilobytes
- B. Number of images generated per day
- C. Downstream product metric: CTR or conversion rate for content using AI-generated images vs. stock images
- D. Model version number of the generation API

_Answers are in `checks.json` in the lesson directory._